<div align="left">

# 第 2 册：第一个算子——从安装到验收的完整体验

本册是零基础篇的核心：你将安装 CANNBot（Flash 版），把一个 add 算子的需求交给它，看着它从环境探测一路走到真机验证，最后**读懂它交付的每一份产物、亲手确认精度证据**。

本册演示的算子与 [《Ascend C 算子开发系列教程（Kernel 直调）》02.04 节](./../../ascendc_operator_development_light/02_AscendC_basic/02.04_introduction_to_kernel_functions_based_on_add_operator.ipynb)手动开发的 **add 算子完全同规格**——`z = x + y`，float32，shape (8, 2048)，核函数名 `add_custom`。学过那门课的读者，可以在本册的各个「📖 对照」卡片里直观看到**手动开发 vs Agent 开发**两条路径的差别；没学过的也不受影响，对照卡片可以当作对那门课的预告。

本册的实操记录来自一次**真实的端到端运行**（Ascend 910B3 + OpenCode + CANN 9.1），所有命令、输出、代码都是真实采集的——你照着做，应该得到同样结构的结果。

路线：**环境自查 → 四步安装 → 需求五要素 → 发出需求 → 观察推进 → 读懂产物 → 验收复验 → 排错指南 → 课后实践**。全程约 40 分钟。

> **环境提示**：涉及真实安装与算子生成，推荐在 [CANNLab 云开发环境](https://gitcode.com/org/cann/cannlab)（预装 NPU 与 CANN，参考[体验指南](https://gitcode.com/cann/cann-learning-hub/blob/master/docs/CANNLab_env_experience_guide.md)）或本地昇腾环境中实践。命令均在**终端**里执行。
</div>

<div align="left">

## 2.1 环境要求自查

<table style="margin-left:0;border-collapse:collapse"><thead><tr><th style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">项目</th><th style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">要求</th><th style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">说明</th></tr></thead><tbody>
<tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">硬件</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">昇腾 NPU（Ascend 910 及后续系列）</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">算子要在真实 NPU 上验证精度</td></tr>
<tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">CANN</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">9.0.0 及以上</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">编译算子内核所必需</td></tr>
<tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">CLI 工具</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">OpenCode / Claude Code / Codex / Trae 等任选其一</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">Agent 宿主，本课程以 OpenCode 为例</td></tr>
<tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">网络 / 磁盘</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">能访问 GitCode；建议 50GB 可用空间</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">安装需克隆仓库与依赖</td></tr>
</tbody></table>

> 📖 OpenCode 安装见[官方文档](https://opencode.ai/docs)；CANN 安装参考[昇腾社区指南](https://www.hiascend.com/document)。没有 NPU 硬件？用 CANNLab 云环境（上方环境提示），后续操作完全一致。

运行下面的代码格自动检查当前环境（支持终端命令的 Notebook 环境可直接跑，也可以复制到终端执行）：
</div>

In [ ]:
import shutil, subprocess

def run(cmd):
    """安全地执行命令：不存在就返回 None。"""
    if shutil.which(cmd.split()[0]) is None:
        return None
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=30)
    return r.stdout.strip() or r.stderr.strip()

print("== 环境自检 ==")
out = run("npu-smi info")
if out:
    first = [l for l in out.splitlines() if l.strip()][2:4]
    print("[OK] NPU 设备：")
    for l in first:
        print("     " + l.strip())
else:
    print("[缺失] 未检测到 npu-smi —— 当前环境没有昇腾 NPU")
    print("       建议：使用 CANNLab 云开发环境，或在有 NPU 的机器上体验")

print()
for tool, tip in [("opencode", "Agent 宿主（OpenCode）"), ("git", "版本管理工具")]:
    ok = shutil.which(tool) is not None
    print(f"[{'OK' if ok else '缺失'}] {tip}：{tool}")
print()
print("提示：CANN 版本可用 `cat /usr/local/Ascend/ascend-toolkit/latest/version.cfg` 查看")

<div align="left">

## 2.2 四步安装 CANNBot（Flash 版）

CANNBot 的 Skills 与 Agents 统一托管在 [cann/cannbot-skills](https://gitcode.com/cann/cannbot-skills) 仓库：

```bash
# 第 1 步：克隆仓库
git clone https://gitcode.com/cann/cannbot-skills.git

# 第 2 步：进入 Flash 插件目录（本课程实操使用的轻量开发流程）
cd cannbot-skills/plugins-official/ops-direct-invoke-flash

# 第 3 步：初始化安装到你的算子工程目录（以 OpenCode 为例）
bash init.sh project opencode ~/my-operators

# 第 4 步：把内置参考算子复制进工程（供 Agent 参照结构）
cp -r operators/{add,mul,sqrt} ~/my-operators/operators/
```

> 💡 `init.sh` 第三个参数是安装目标目录（默认当前目录）。**建议装到独立工程目录**（如 `~/my-operators`），让工作流产物都落在你自己的工作区。
>
> 第 4 步复制的 `add` 参考算子马上会派上用场：2.4 节我们让 CANNBot 开发一个**同规格但不同名**的 `add_custom`，它的第一步就是阅读工程内已有的参考算子来了解结构——这正是「参考算子」机制的意义（新算子 `add_custom` 与参考算子 `add` 名字错开，工作流要求新算子不能与已有算子重名）。

下面是一次**真实安装**的输出（共 4 步，轻量）：

```text
[1/4] Installing skills and agents...
  ✓ Skills: 3 linked（ops-direct-invoke-flash 及配套测试设计技能）
  ✓ Agents: 1 linked（代码评审 Agent）
[2/4] Installing configuration...
  ✓ AGENTS.md（CANNBot Flash 版团队配置）
[3/4] Writing manifest...
  ✓ Manifest: ~/my-operators/.opencode/cannbot-manifest.json
[4/4] Health check...
  ✓ All checks passed
✓ ops-direct-invoke-flash installed successfully!
```

安装后的工程目录长这样：

```text
~/my-operators/
├── AGENTS.md            # CANNBot（Flash 版）团队配置：进入目录即生效
├── .opencode/
│   ├── skills/          #   3 个 Skill（开发工作流 + 测试设计）
│   ├── agents/          #   1 个评审 Agent
│   └── cannbot-manifest.json
└── operators/           # 你的算子工作区
    ├── add/  mul/  sqrt/   # 参考算子（内核 .asc + 测试 + CMake，可照抄结构）
```

> 📖 更完整的安装方式（全局安装、Windows、install-helper 向导）见 [CANNBot 安装指南](https://gitcode.com/cann/cannbot-skills/blob/master/docs/installation-guide.md)。

验证安装（在工程目录下运行下面的代码格）：
</div>

In [ ]:
import os

print("== 安装验证 ==")
print("（在你的算子工程目录（如 ~/my-operators）下执行）")
print()
checks = [
    ("AGENTS.md",                     "CANNBot Flash 版团队配置"),
    (".opencode/skills/",             "Skill 目录（3 个）"),
    (".opencode/agents/",             "Agent 目录（1 个评审 Agent）"),
    (".opencode/cannbot-manifest.json", "安装清单"),
    ("operators/add/add.asc",         "参考算子（add 内核）"),
]
found_any = False
for path, desc in checks:
    ok = os.path.exists(path)
    found_any = found_any or ok
    print(f"[{'✓' if ok else '✗'}] {path:34s} {desc}")

if not found_any:
    print()
    print("当前 Notebook 目录未检测到安装产物——这是正常的：")
    print("安装发生在你自己的算子工程目录中，完成 2.2 节后在那边重新运行本格。")

<div align="left">

## 2.3 把需求说清楚：任务定义五要素

「帮我写个 add」太模糊——模型会自行补全 shape、dtype，每次结果都不一样，你也无从判断对错。一份好的任务定义要说清五组信息，且每一组都能在后续产物中找到对应：

<table style="margin-left:0;border-collapse:collapse"><thead><tr><th style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">信息组</th><th style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">要回答的问题</th><th style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">对应产物</th></tr></thead><tbody>
<tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">名称与公式</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">算子叫什么？输出如何由输入计算？</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">定义文档、内核代码</td></tr>
<tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">输入与输出</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">参数名、shape、dtype 是什么？</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">定义文档、host 入口签名</td></tr>
<tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">支持范围</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">固定 shape 还是动态？覆盖哪些边界？</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">测试用例矩阵</td></tr>
<tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">参考与测试</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">拿什么当标准答案？在哪个设备上测？</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">测试脚本（CPU 参考对比）</td></tr>
<tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">停止条件</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">做到哪一步算完成？</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">STATE.md 的验收记录</td></tr>
</tbody></table>

> 📖 **对照①：五要素的「人肉版」你其实见过。** 手动开发同一个 add 算子时（[light 教程 02.04 节](./../../ascendc_operator_development_light/02_AscendC_basic/02.04_introduction_to_kernel_functions_based_on_add_operator.ipynb)），第一步「算子分析」要依次明确：数学表达式 $\vec{z} = \vec{x} + \vec{y}$；输入输出（x/y 输入、z 输出，float，(8, 2048)，ND）；核函数名与参数（`add_custom`，x/y/z 的 GM 地址）；所需 API（DataCopy / Add / AllocTensor / EnQue）——这四步本质上就是五要素的展开。区别在于：手动开发要**自己**把分析逐行落成 TilingData 结构体、算子类和 host 代码；而这里你只需把分析**说清楚**，落地的活交给 Agent。

运行代码，用「五要素检查器」检验两份任务描述——一份含糊、一份合格：
</div>

In [ ]:
def check_task(task: str) -> list:
    """检查任务描述是否覆盖五要素，返回缺失项列表。"""
    rules = [
        ("名称与公式", ["add", "+", "="]),
        ("输入与输出", ["float32", "fp32", "输入"]),
        ("支持范围",  ["shape", "(8, 2048)", "2048"]),
        ("参考与测试", ["验证", "精度", "NPU"]),
        ("停止条件",  ["通过后", "即停止", "验收"]),
    ]
    missing = [name for name, kws in rules if not any(k in task for k in kws)]
    return missing

tasks = {
    "含糊版": "帮我写个 add",
    "合格版": ("使用 ops-direct-invoke-flash 开发一个 add_custom 算子：z = x + y，逐元素相加，"
               "输入 x 和 y shape (8, 2048)，float32，输出 z 与输入同 shape 同 dtype。"
               "在真实 NPU 上验证精度，精度验收通过后即停止。"),
}
for label, task in tasks.items():
    missing = check_task(task)
    print(f"【{label}】{task[:40]}...")
    if missing:
        print(f"  缺失：{'、'.join(missing)}  ← Agent 会自由发挥这些空隙\n")
    else:
        print(f"  五要素齐全 ✓  ← 每一项都能在产物中找到对应\n")

<div align="left">

## 2.4 发出需求：环境探测与档位判定

进入你的算子工程目录，启动 `opencode`，把合格版任务发送出去。CANNBot 的第一步是**搞清楚自己在什么硬件上干活**——下面是真实运行的记录：

```text
$ python3 .opencode/skills/ops-direct-invoke-flash/scripts/detect_soc.py
SocVersion: Ascend910B3
NpuArch:    dav-2201

目标芯片：Ascend910B3（dav-2201），非 dav-3510，走经典 AscendC 计算路径。
add_custom 为纯逐元素运算，判定为 simple 档；harness.review=auto + simple
→ 跳过文档评审 Agent；harness.test_gate=off → 跳过阶段 7.5。
结构参考：阅读工程内参考算子 operators/add；确认 add_custom.asc 尚不存在（与参考算子不重名）。
```

四件事在这一步发生了：

1. **芯片探测**：识别出 Ascend910B3（架构 dav-2201）——不同架构代码生成路径不同（Ascend 950 的 dav-3510 走新一代 Reg API 路径）；
2. **复杂度分档**：add_custom 是纯逐元素运算，判为 **simple 档**——评审环节自动精简，换取速度（复杂算子会走完整的多 Agent 评审）；
3. **选结构参照**：阅读 2.2 节复制进来的参考算子 `add`，确认新算子 `add_custom` 与它不重名——Agent 不从零发明结构，而是站在已验证的样板肩上；
4. **建档**：创建 `docs/add_custom/STATE.md` 作为进度唯一可信来源，后续每阶段完成都勾选记录。

这就像靠谱的工程师开工前先确认机床型号、看图纸难度、找一份同类图纸做参照、立项目台账——而不是上来就猛写代码。

## 2.5 观察推进：八个阶段逐步落地

接下来 CANNBot 按流程推进。对话里你会看到它创建的任务清单：

```text
[✓] 阶段1: 创建 docs/add_custom/ 目录结构与 STATE.md
[✓] 阶段2: 创建骨架 add_custom.asc + CMakeLists.txt + 占位测试，仅 cmake 配置
[✓] 阶段3: 编写 add_custom_definition.md（跳过评审并记录）
[✓] 阶段4: 编写 add_custom_design.md（含确定性校验，跳过评审并记录）
[✓] 阶段5: 编写完整测试套件（py_compile 静态检查）
[✓] 阶段6: 实现完整核函数，唯一一次构建 + 全量真机测试
[✓] 阶段7: NPU 验证（simple 档跳过干净重建，确认实现完整性与日志）
[✓] 阶段7.5: 记录跳过测试门禁（test_gate=off）
[✓] 阶段8: 收尾文档与最终报告
```

<img src="./images/02_eight_stage_workflow.png" alt="八阶段工作流推进示意图" width="800px">

注意几个「工程味」的细节：

- **骨架先行**：先搭能编译的空壳验证工具链，再往里填肉；
- **文档先行**：先写定义/设计文档，语义想清楚再动手；
- **测试先行**：测试套件在实现之前写好，实现完成立刻被 17 个用例检验——TDD 的节奏；
- **跳过也留痕**：每个被跳过的环节都在 STATE.md 记录原因——「跳过」不等于「没有交代」。

> 📖 **对照②：Tiling 不用你算了。** 手动开发时，02.04 节要自己定义 `AddCustomTilingData` 结构体（totalLength、tileNum）、推导 `blockLength = totalLength / 核数`、斟酌 `BUFFER_NUM = 2` 的双缓冲时序。这次 CANNBot 在阶段 4 的设计文档里自动完成了同样的推导：查询可用核数（GetCoreNumAiv）→ 按「每核最少 1024 元素」约束算出 numBlocks=16、blockLength=1024（对 (8, 2048) 共 16384 个元素）→ 按 UB 容量（196608 字节）除以队列数与流水深度算出每 tile 的字节数。手动课程里最容易出错的一环，在这里变成了一页可检视的算术。

本次实测全程约 11 分钟（含文档与测试生成；不同模型与网络条件下约 10~30 分钟）。期间可以盯着对话看它一步步干活——每条命令、每个文件写入都实时可见。
</div>

<div align="left">

## 2.6 读懂产物：operators/add_custom/ 里有什么

任务完成后，`operators/add_custom/` 下是一套完整的算子工程。运行下面的代码查看结构（若你已在自己环境完成实操，会看到真实目录；否则展示本次实测的结构）：
</div>

In [ ]:
import os

REAL = "operators/add_custom"
demo_tree = """operators/add_custom/                ← 算子工程根目录
├── add_custom.asc                   # 内核代码（148 行：device kernel + tiling + host 入口 + 注册）
├── test_add_custom.py               # 测试套件（17 个用例）
├── CMakeLists.txt                   # 构建配置（产出 libop_add_custom.so）
├── build/                           # 构建产物目录（make 生成）
└── docs/add_custom/
    ├── STATE.md                     # 进度台账（每阶段勾选记录）
    ├── add_custom_definition.md     # 定义文档：数学语义 / dtype 策略 / 边界场景
    ├── add_custom_design.md         # 设计文档：tiling 方案 / UB 预算 / 确定性校验
    └── plans/                       # 排障记录目录
"""

if os.path.isdir(REAL):
    print("检测到本地 " + REAL + "，真实目录结构：")
    for root, dirs, files in os.walk(REAL):
        dirs[:] = [d for d in dirs if d not in ("build", "__pycache__", ".pytest_cache")]
        depth = root.count(os.sep)
        print("  " * depth + os.path.basename(root) + "/")
        for f in sorted(files):
            print("  " * (depth + 1) + f)
else:
    print("本次实测的产物结构（在你自己的环境完成实操后会生成同样结构）：")
    print(demo_tree)

<div align="left">

### 走读 add_custom.asc：148 行的完整内核

`.asc` 是 Ascend C 的单文件算子源码格式，一个文件装下四层结构（节选自实测代码，注释为讲解所加）：

```cpp
// ① Device 内核：三级流水（每核认领一段连续数据，逐 tile 处理）
template <typename T>
__global__ __aicore__ void add_custom_kernel(GM_ADDR x, GM_ADDR y, GM_ADDR z,
                                             int64_t totalLength, int64_t blockLength, uint32_t tileSize)
{
    AscendC::TPipe pipe;                                    // 片上内存管线
    AscendC::TQue<AscendC::QuePosition::VECIN,  2> inQueueX, inQueueY;   // 双缓冲输入队列
    AscendC::TQue<AscendC::QuePosition::VECOUT, 2> outQueueZ;           // 双缓冲输出队列
    // ... xGm/yGm/zGm 各自按核序号偏移到本核段起点 ...
    for (int64_t i = 0; i < tileNum; ++i) {
        AscendC::DataCopyPad(xLocal, xGm[offset], copyParams, padParams);  // GM → UB（搬入 x）
        AscendC::DataCopyPad(yLocal, yGm[offset], copyParams, padParams);  // GM → UB（搬入 y）
        AscendC::Add(zLocal, xLocal, yLocal, elementNumPerTile);           // 核心计算：一条向量加法
        AscendC::DataCopyPad(zGm[offset], zLocal, copyParams);             // UB → GM（搬出 z）
    }
    if (tailTileElementNum > 0) { /* 尾块处理：非对齐部分单独搬运 */ }
}

// ② Tiling：决定数据怎么切
std::tuple<int64_t, int64_t, int64_t> calc_add_custom_tiling_params(int64_t totalLength)
{
    constexpr static int64_t MIN_ELEMS_PER_CORE = 1024;   // 每核最少 1024 个元素
    int64_t coreNum = platform->GetCoreNumAiv();          // 查询可用核数
    int64_t numBlocks = std::min(coreNum, (totalLength + 1023) / 1024);
    int64_t tileSize = ubSize / PIPELINE_DEPTH / BUFFER_NUM;  // 按 UB 容量算 tile 大小
    return {numBlocks, blockLength, tileSize};
}

// ③ Host 入口：PyTorch 调用的桥梁
namespace op_add_custom {
at::Tensor add_custom(const at::Tensor& x, const at::Tensor& y)
{
    auto aclStream = c10_npu::getCurrentNPUStream().stream(false);
    at::Tensor z = at::empty(x.sizes(), x.options());
    auto [numBlocks, blockLength, tileSize] = calc_add_custom_tiling_params(x.numel());
    add_custom_kernel<float><<<numBlocks, nullptr, aclStream>>>(/*...*/);  // <<<>>> 核函数直调
    return z;
}
}

// ④ torch 注册：让 Python 能调用
TORCH_LIBRARY(op_add_custom, m)            { m.def("add_custom(Tensor x, Tensor y) -> Tensor"); }
TORCH_LIBRARY_IMPL(op_add_custom, PrivateUse1, m) { m.impl("add_custom", TORCH_FN(op_add_custom::add_custom)); }
```

四层各司其职：**内核**管「怎么算」（三级流水，多核各算一块，尾块单独处理）；**Tiling** 管「怎么切」（查核数、按 1024 元素下限分块、按 UB 容量定 tile）；**Host 入口**管「怎么被调用」（`<<<>>>` 直调语法下发内核）；**注册**管「怎么进 PyTorch」（之后 Python 里 `torch.ops.op_add_custom.add_custom(x, y)` 就能用）。

> 📖 **对照③：跟手动开发的 KernelAdd 是同一副骨架。** 02.04 节手动实现的算子类 `KernelAdd`：`Init()` 做核间切分与 `SetGlobalBuffer` 偏移，`Process()` 跑 CopyIn→Compute→CopyOut 三级流水，`BUFFER_NUM = 2` 双缓冲。CANNBot 生成的核函数遵循完全相同的范式——TQue 深度 2（双缓冲）、每核认领 blockLength 连续段、主循环逐 tile「搬入-计算-搬出」、尾块单独走 DataCopyPad。手动课程教的矢量编程范式，正是 Agent 生成代码时遵循的范式；学过 02.04 的你读这份产物应该毫不费力。

<img src="./images/02_pipeline_double_buffer.png" alt="三级流水与双缓冲示意图" width="800px">

> 💡 暂时看不懂每行 C++ 也没关系——重点是建立「一份内核代码包含哪几层」的地图感。想系统学 Ascend C 语法，见[相关课程](./../README.md)。

### 走读 test_add_custom.py：17 个用例怎么来

测试是一个**用例矩阵**（节选自实测代码）：

```python
SHAPES = [
    (8, 2048),    # 用户指定 shape
    (1,), (3,), (100,), (1024,), (10000,),   # 1D：从最小到大
    (7, 333),     # 2D，非 tile 对齐（考验尾块处理）
    (256, 512),   # 2D
    (5, 10, 15),  # 3D
    (2, 3, 32, 32),  # 4D
    (1000, 1000), (999, 999),  # 大 shape：覆盖主 tile 循环
]
DTYPES = [torch.float32]

@pytest.mark.skipif(not torch.npu.is_available(), reason="NPU device not found")
@pytest.mark.parametrize("shape", SHAPES)
@pytest.mark.parametrize("dtype", DTYPES)
def test_add_custom_operator(shape, dtype):
    x = torch.randn(*shape, dtype=dtype)
    y = torch.randn(*shape, dtype=dtype)
    result = torch.ops.op_add_custom.add_custom(x.npu(), y.npu()).cpu()
    expected = x + y                             # CPU 参考实现 = 标准答案
    # fp32 加法是 IEEE 754 精确运算（单次舍入），要求位级一致
    assert torch.allclose(result, expected, rtol=0.0, atol=0.0)
```

除参数化用例外，还有 4 个**边界值专项**（±0 与带符号零、±Inf 混合、NaN 语义、有限最大值溢出）和 1 个接口存在性测试，合计 17 个。

三个值得学的测试设计：

1. **独立参考**：标准答案来自 CPU 的 `x + y`——不是「实现跟自己比」；
2. **边界值专项**：除随机数据外，还有用例覆盖 ±0（含符号位保持）、±Inf（Inf+(-Inf)=NaN）、NaN、fp32 最大有限值相加溢出到 Inf；
3. **容差按语义定**：fp32 逐元素加法只有一次舍入、无超越函数，NPU 与 CPU 理论上位级一致，所以容差 `rtol=0.0, atol=0.0`（NaN 位置按语义比对）——容差不是拍脑袋，是算子语义推出来的。

> 📖 **对照④：交付件从「两件套」变「四件套」。** 手动开发 02.04 的交付件是 `add_custom.asc`（内核 + host 验证 main 的单文件）加 `CMakeLists.txt`，验收靠 main 里手写的 golden 比对，跑完即弃。CANNBot 的交付件多了两样：**独立于实现的 `test_add_custom.py`**（17 个用例的可复跑回归资产）与 **`docs/` 台账**（定义/设计/进度，记录「为什么这么写」）。验收也从「跑一次 main」变成「pytest 与 CPU 参考逐用例自动比对」。
</div>

<div align="left">

## 2.7 验收证据：17/17 与「位级精确」

阶段 6 的构建 + 真机测试是全流程的正确性关卡。真实运行输出：

```text
$ cd operators/add_custom/build && make -j
$ cd .. && pytest test_add_custom.py -v
test_add_custom.py::test_add_custom_interface_exist PASSED                [  5%]
test_add_custom.py::test_add_custom_operator[dtype0-shape0] PASSED       [ 11%]
  ...（15 个用例）...
test_add_custom.py::test_add_custom_boundary_values[max_finite_overflow-...] PASSED [100%]

======================== 17 passed, 1 warning in 10.70s ========================
```

CANNBot 最终汇报里的验收结论：

> **测试套件：17/17 通过**（10.70s）；用户指定 shape：`(8, 2048)` fp32 结果**位级精确**（bit-exact: True，max_diff = 0.0），shape/dtype 完全保持。

「位级精确」意味着 NPU 输出和 CPU 参考的每一个比特都相同——对 fp32 加法这种单次舍入的精确运算，这是最严格也最合理的要求。**这就是第 1 册说的「闭环」的终点：不是 Agent 说自己做完了，而是有可复现的测试证据。**

不要只信汇报——自己跑一遍。如果你已在自己环境完成实操，运行下面的代码复验：
</div>

In [ ]:
import subprocess, shutil, os

if os.path.isdir("operators/add_custom") and shutil.which("pytest"):
    print("检测到本地 operators/add_custom，运行复验…\n")
    r = subprocess.run(["pytest", "test_add_custom.py", "-v"],
                       cwd="operators/add_custom", capture_output=True, text=True, timeout=600)
    tail = "\n".join(r.stdout.splitlines()[-6:])
    print(tail)
else:
    print("本地未检测到 operators/add_custom——请先完成 2.4~2.5 的实操，再回来复验。")
    print()
    print("复验命令备忘（在算子工程目录下执行）：")
    print("  cd operators/add_custom && pytest test_add_custom.py -v")
    print()
    print("期望结果：17 passed——与 2.7 节的验收证据一致。")

<div align="left">

## 2.8 出错了怎么办

真实世界不会永远一次通过。常见三类情况：

<table style="margin-left:0;border-collapse:collapse"><thead><tr><th style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">情况</th><th style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">典型表现</th><th style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">怎么办</th></tr></thead><tbody>
<tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">编译失败</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">make 报错、符号未定义</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">把报错贴回对话，CANNBot 会自己修（迭代循环就是干这个的）</td></tr>
<tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">精度不过</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">部分用例 FAILED、max_diff > 0</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">同样贴回报错；它会按「症状→原因→排查」的调试技能定位（第 3 册详述）</td></tr>
<tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">环境问题</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">找不到 NPU、CANN 版本报错</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">按 2.1 自查环境；升 CANN 或换 CANNLab 环境重试</td></tr>
</tbody></table>

安装环节的常见问题：**安装报错**→检查 CANN 版本与网络（能否访问 GitCode）；**查看帮助**→`bash init.sh --help`；**更新**→`git pull` 后重跑 `bash init.sh`（幂等，已有配置保留）；**Agent 不认识算子需求**→确认在 init 过的目录里启动（该目录下有 `AGENTS.md`）。

记住一个原则：**CANNBot 的每一次迭代都留下了记录**（STATE.md、对话、测试输出）。出问题不可怕，可怕的是没有证据链的瞎改——保留现场，贴给 Agent，让它基于事实修。
</div>

<div align="left">

## 2.9 课后练习

本册带你完成了从安装、需求定义到真机验收的第一次算子交付，请根据学习内容完成以下题目自测；第 4、5 题为编程实践题，请独立完成。

1. （单选题）任务定义五要素中，「停止条件」的作用是？  
    A. 装饰性的，可有可无  
    B. 明确做到哪一步算完成（如「精度验收通过后即停止」），避免 Agent 跑偏做过头  
    C. 限定运行时长  
    D. 指定使用的编程语言

2. （多选题）本次 add_custom 实测中，测试套件的设计包含哪些要点？  
    A. 独立 CPU 参考实现作标准答案  
    B. 用例矩阵覆盖用户指定 shape + 非对齐 + 多维  
    C. 边界值专项用例（±0 与带符号零、±Inf、NaN、溢出）  
    D. 容差全部使用默认值 rtol=1e-3

3. （单选题）`.asc` 单文件内核中的 Tiling 部分（`calc_add_custom_tiling_params`）负责什么？  
    A. 注册算子到 PyTorch  
    B. 决定数据怎么切分（核数、每块长度、tile 大小）  
    C. 执行真正的加法计算  
    D. 生成测试用例

4. （编程实践·必做）用 CANNBot 开发你的第二个算子：`sigmoid_custom`（入门级）。  
   仿照 2.3~2.7 的完整流程，把下面的算子规格交给 CANNBot。**先别急着发需求**——对照 2.3 的五要素，自己把它扩写成一份合格的任务定义（提示：想想 sigmoid 与 add 的关键差异：它含 exp 超越函数，精度口径该怎么表述？）。  
   算子规格：  
   - 计算公式：`sigmoid(x) = 1/(1 + exp(-x))`，逐元素运算  
   - 数据类型：输入与输出均为 float32  
   - 数据形状：输入 shape (8, 2048)，输出与输入相同  
   - 实现约束：**禁止直接使用 `AscendC::Sigmoid` 高阶指令**。add 用一条 `AscendC::Add` 指令就够；sigmoid 其实也有一条 `AscendC::Sigmoid` 指令，但本题要求把公式拆开，用 `Muls`、`Exp`、`Adds`、`Div` 等基础向量指令组合实现  
   - 验收标准：pytest 全部用例通过；需求描述覆盖五要素；打开生成的 `sigmoid_custom.asc` 检查确认没有 `AscendC::Sigmoid` 调用——学会验证 Agent 遵守了你的约束

5. （编程实践·选做）把 `sigmoid_custom` 扩展为泛化版（进阶级，对齐手动课程 [02.09 实践](./../../ascendc_operator_development_light/02_AscendC_basic/02.09_vector_operator_practice.ipynb)）。  
   在必做完成的基础上，继续向 CANNBot 追加一条扩展需求，使算子支持 float16/float32 双 dtype 输入输出，且测试用例矩阵必须覆盖三个指定 case（实现约束同第 4 题）：  
   - case1：输入 x 的 shape 为 `(7, 83)`，float32  
   - case2：输入 x 的 shape 为 `(1024, 1024)`，float16  
   - case3：输入 x 的 shape 为 `(999, 999)`，float32  
   提交物（第 4、5 题）：你写的需求描述原文（两级）、`operators/sigmoid_custom/` 产物目录、pytest 验收输出。

**执行以下代码获取答案。**

</div>

In [ ]:
!cat ./answer/02_answer.txt

<div align="left">

## 本册小结

- **环境与安装**：三要素（NPU / CANN ≥ 9.0.0 / CLI 工具）；四步安装（clone → init.sh → 复制参考算子 → 验证）；
- **需求五要素**：名称公式 / 输入输出 / 支持范围 / 参考测试 / 停止条件——每一项都能在产物中找到对应；
- **工作流推进**：环境探测（芯片/档位/参照）→ 骨架 → 文档 → 测试 → 实现 → 真机 → 收尾；骨架先行、文档先行、测试先行、跳过留痕；
- **产物地图**：`add_custom.asc` 四层一体（内核/tiling/host 入口/注册）、`test_add_custom.py` 用例矩阵、`docs/` 台账；
- **验收证据**：17/17 通过、位级精确（max_diff=0.0）——不信汇报信证据，亲手复验；
- **课后实践**：用 `sigmoid_custom` 独立走一遍「需求 → 交付」（必做），再扩展为 float16/float32 泛化版（选做）。

你已经完整走过一次「需求 → 交付」，并具备独立发起第二次的能力。最后一册：[玩转与共建——日常场景、评测标尺与社区入口](./03_capabilities_and_community.ipynb)。
</div>